# Chapter 15 — Production Inference Economics

**Math for ML Engineers** · companion notebook

This notebook mirrors §15.2 (the roofline model) and §15.4 (KV-cache sizing) of the chapter, turning the arithmetic-intensity argument and the memory-bandwidth accounting behind LLM serving costs into runnable code. **Key takeaway: at large batch size, autoregressive decoding is memory-bandwidth-bound, not compute-bound — the GPU spends nearly all its time waiting on HBM to deliver weights, and that single fact is the reason inference serving systems are built the way they are.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

## The roofline model

Every kernel that runs on an accelerator moves some number of bytes and performs some number of floating-point operations. Its **arithmetic intensity** (AI) is the ratio of the two:

$$
\text{AI} = \frac{\text{FLOPs}}{\text{bytes moved}} \qquad \text{[FLOPs/byte]}
$$

A GPU has two hard ceilings: a peak compute rate $P_{\text{flops}}$ (FLOPs/s) and a peak memory bandwidth $P_{\text{bw}}$ (bytes/s). The best throughput any kernel can achieve, in FLOPs/s, is the smaller of "as fast as the chip can compute" and "as fast as the chip can feed it data":

$$
T(\text{AI}) = \min\big(P_{\text{flops}},\ \text{AI} \cdot P_{\text{bw}}\big)
$$

Plotted on log-log axes against AI, this traces a **roofline**: a rising line of slope 1 (memory-bound region) that flattens into a horizontal plateau at $P_{\text{flops}}$ (compute-bound region). The two segments meet at the **ridge point** $\text{AI}^\star = P_{\text{flops}} / P_{\text{bw}}$ — the arithmetic intensity a kernel needs to just barely saturate compute. Below the ridge point, buying more FLOPs does nothing; the kernel is starved for bytes.

For an NVIDIA A100: $P_{\text{flops}} = 312$ TFLOPS (bf16 tensor cores), $P_{\text{bw}} = 2$ TB/s (HBM2e). That puts the ridge point at $\text{AI}^\star = 312\text{e}12 / 2\text{e}12 = 156$ FLOPs/byte — a kernel needs to do at least 156 floating-point operations per byte it touches before more compute helps at all.

In [ ]:
def arithmetic_intensity(flops, bytes_moved):
    """AI = FLOPs / bytes."""
    return flops / bytes_moved


def roofline(ai, peak_flops, peak_bw):
    """Achievable throughput under the roofline model."""
    return min(peak_flops, ai * peak_bw)


def tpot_memory_bound(weight_bytes, peak_hbm_bw):
    """TPOT (time per output token) when memory-bound = total_weight_bytes / HBM_bandwidth."""
    return weight_bytes / peak_hbm_bw


# A100 SXM specs (bf16 tensor cores, HBM2e)
PEAK_FLOPS = 312e12   # FLOPs/s
PEAK_BW = 2e12         # bytes/s
RIDGE_AI = PEAK_FLOPS / PEAK_BW

print(f"A100 peak compute:    {PEAK_FLOPS/1e12:.0f} TFLOPS (bf16)")
print(f"A100 peak HBM bw:     {PEAK_BW/1e12:.0f} TB/s")
print(f"Ridge point AI*:      {RIDGE_AI:.0f} FLOPs/byte")

## Prefill vs. decode: two very different arithmetic intensities

A transformer forward pass has two distinct phases in serving, and they sit on opposite sides of the ridge point:

- **Prefill** processes the whole prompt at once. The same weights are reused across every token in the prompt, so FLOPs scale with (prompt length × params) while weight bytes moved stay roughly fixed — arithmetic intensity is high, and with enough tokens in flight, prefill can approach the compute-bound plateau.
- **Decode** generates one token at a time. Each step still has to read *every* parameter of the model — for a 70B-parameter model, decode literally streams 70B parameters through the chip per output token — but performs only $O(1)$ token's worth of FLOPs per parameter read. Arithmetic intensity is low (well under the ridge point), so decode is **memory-bandwidth-bound**: `tpot_memory_bound` says the minimum time per output token is just `weight_bytes / HBM_bandwidth`, independent of how fast the compute units are.

This is also where the KV-cache enters. To avoid recomputing attention over the whole prefix at every decode step, the model caches the key and value projections for every past token. For $L$ layers, $n_{kv}$ KV heads (equal to `K_heads`, or reduced under grouped-query attention to `K_groups`), head dimension $d_{head}$, and $P$ cached tokens, at `dtype_bytes` bytes per element:

$$
\text{KV bytes} = 2 \cdot L \cdot n_{kv} \cdot d_{head} \cdot P \cdot \text{dtype\_bytes}
$$

(the factor of 2 is for storing both K and V). This cache lives in the same HBM that decode is already bandwidth-starved for, and it grows *linearly* with context length — which is why long-context serving becomes a memory-capacity problem well before it becomes a compute problem.

In [ ]:
def kv_cache_bytes(L_layers, K_heads, d_head, P_tokens, dtype_bytes=2, K_groups=None):
    """KV-cache size for one request. n_kv_heads = K_heads or K_groups (GQA). Returns bytes for K+V."""
    n_kv_heads = K_heads if K_groups is None else K_groups
    return 2 * L_layers * n_kv_heads * d_head * P_tokens * dtype_bytes


def llama3_70b_kv():
    """Spec: 80 layers, 64 attn heads, 8 KV groups, d_head=128, bf16."""
    return dict(L_layers=80, K_heads=64, d_head=128, K_groups=8, dtype_bytes=2)


def speculative_speedup(alpha, gamma, c=1.0):
    """Leviathan-Kalai: E[tokens/call]/E[walltime/call] = (1-alpha^(gamma+1)) / ((1-alpha)(c*gamma+1))"""
    num = 1 - alpha ** (gamma + 1)
    den = (1 - alpha) * (c * gamma + 1)
    return num / den


def quantize_int8_perchannel(W, axis=0):
    """Symmetric per-channel INT8 quantizer. Returns (W_int8, scales)."""
    abs_max = np.max(np.abs(W), axis=axis, keepdims=True)
    scales = abs_max / 127.0
    W_int = np.round(W / np.where(scales == 0, 1, scales)).clip(-128, 127)
    return W_int.astype(np.int8), scales


def dequantize(W_quant, scales):
    return W_quant.astype(np.float32) * scales


# 70B model, bf16 weights: decode TPOT under pure memory-bound assumption
n_params_70b = 70e9
weight_bytes_70b = n_params_70b * 2  # bf16 = 2 bytes/param
tpot_70b = tpot_memory_bound(weight_bytes_70b, PEAK_BW)

kv_spec = llama3_70b_kv()
kv_128k = kv_cache_bytes(**kv_spec, P_tokens=128_000)

print(f"70B model weight bytes:      {weight_bytes_70b/1e9:.1f} GB")
print(f"Memory-bound decode TPOT:    {tpot_70b*1e3:.2f} ms/token  (best case, single request)")
print(f"Llama-3-70B KV-cache @128K:  {kv_128k/1e9:.1f} GB per request")

## Visualizing the roofline and the KV-cache blowup

Two plots make the economics legible:

1. **The roofline itself** — arithmetic intensity on the x-axis (log scale), achievable throughput on the y-axis (log scale), with the memory-bound rising segment meeting the compute-bound plateau at the A100's ridge point. We mark two operating points for the 70B model: a **prefill** point at a plausible high-batch AI (well to the right of the ridge, near the compute plateau) and a **decode** point at the AI implied by streaming all 70B parameters per token (far to the left, deep in the memory-bound region).
2. **KV-cache size vs. context length** — `kv_cache_bytes` for the Llama-3-70B GQA configuration evaluated from 1K to 128K tokens of context, log-log, with the 128K / ~40GB point annotated directly on the curve.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Roofline plot ---
ai_range = np.logspace(-1, 4, 400)
throughput = np.minimum(PEAK_FLOPS, ai_range * PEAK_BW)

axes[0].plot(ai_range, throughput, color="tab:blue", linewidth=2, label="A100 roofline")
axes[0].axvline(RIDGE_AI, color="gray", linestyle=":", linewidth=1)
axes[0].text(RIDGE_AI * 1.15, PEAK_FLOPS * 0.08, f"ridge AI*={RIDGE_AI:.0f}", color="gray", fontsize=9)

# Decode operating point: AI implied by reading all params, ~1 FLOP-pair per param per token
decode_ai = arithmetic_intensity(flops=2 * n_params_70b, bytes_moved=weight_bytes_70b)
decode_throughput = roofline(decode_ai, PEAK_FLOPS, PEAK_BW)

# Prefill operating point: large batch pushes AI well past the ridge, toward the compute plateau
prefill_ai = RIDGE_AI * 20
prefill_throughput = roofline(prefill_ai, PEAK_FLOPS, PEAK_BW)

axes[0].scatter([decode_ai], [decode_throughput], color="tab:red", zorder=5, s=90, label="decode (70B, batch=1)")
axes[0].scatter([prefill_ai], [prefill_throughput], color="tab:green", zorder=5, s=90, label="prefill (large batch)")

axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("arithmetic intensity [FLOPs/byte]")
axes[0].set_ylabel("achievable throughput [FLOPs/s]")
axes[0].set_title("Roofline: A100, bf16")
axes[0].legend(loc="lower right", fontsize=9)
axes[0].grid(True, which="both", alpha=0.2)

# --- KV-cache vs context length ---
context_lengths = np.logspace(np.log10(1_000), np.log10(128_000), 200)
kv_sizes_gb = np.array([kv_cache_bytes(**kv_spec, P_tokens=p) for p in context_lengths]) / 1e9

axes[1].plot(context_lengths, kv_sizes_gb, color="tab:purple", linewidth=2)
axes[1].scatter([128_000], [kv_128k / 1e9], color="tab:red", zorder=5, s=90)
axes[1].annotate(f"128K ctx -> {kv_128k/1e9:.1f} GB",
                  xy=(128_000, kv_128k / 1e9), xytext=(15_000, kv_128k / 1e9 * 0.55),
                  arrowprops=dict(arrowstyle="->", color="gray"), fontsize=9)

axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("context length [tokens]")
axes[1].set_ylabel("KV-cache size per request [GB]")
axes[1].set_title("Llama-3-70B KV-cache growth (GQA, 8 groups)")
axes[1].grid(True, which="both", alpha=0.2)

plt.tight_layout()
plt.show()

print(f"Decode AI:  {decode_ai:.2f} FLOPs/byte  ({decode_ai/RIDGE_AI:.4f}x the ridge point -- deep in the memory-bound region)")
print(f"Prefill AI: {prefill_ai:.0f} FLOPs/byte  ({prefill_ai/RIDGE_AI:.1f}x the ridge point -- near the compute plateau)")
print(f"KV-cache at 128K context: {kv_128k/1e9:.2f} GB per request (single request, before any batching)")

## Why this gap is why vLLM and TensorRT-LLM physically split prefill from decode

The roofline plot above is not an academic curiosity — it is a direct explanation for one of the biggest architectural shifts in LLM serving over the past two years: **disaggregated serving**, where prefill and decode run as separate services on separate GPU pools (used in production by vLLM's disaggregated prefill, and by TensorRT-LLM / Dynamo-style deployments).

The reason is exactly the gap between the green and red dots above. Prefill wants **large, compute-dense batches**: more concurrent prompts push arithmetic intensity to the right, toward the compute plateau, so the GPU's FLOPs are actually the bottleneck being paid for. Decode wants the opposite — it is *already* memory-bound at batch size 1 (that red dot sits at roughly `decode_ai / RIDGE_AI` of the ridge point, nowhere close to needing more compute), so the only lever that helps decode throughput is **more parallel requests sharing the same weight read** (continuous batching), not more FLOPs. Colocating the two phases on one GPU pool forces a bad compromise: prefill requests either wait behind decode's slow, bandwidth-starved steps, or decode's tight per-token latency budget gets blown by a prefill batch hogging the compute units. Splitting them onto separate pools lets each be scheduled — and each pool's hardware provisioned — for the resource it actually needs.

The KV-cache plot ties in directly. Because `kv_cache_bytes` grows **linearly** in context length `P_tokens`, and that cache must live in the same HBM capacity that decode already depends on for weight streaming, long-context serving is fundamentally a **memory-capacity problem**, not a compute problem — a 128K-context request can burn ~40 GB of HBM before a single extra FLOP is spent on it. This is precisely why grouped-query attention (GQA) exists as an architectural choice at all: the `K_groups=8` parameter in `kv_cache_bytes` is a *direct, linear* lever on cache size — going from 64 independent KV heads (`K_heads`) to 8 shared groups cuts the KV-cache by 8x with only a modest quality cost, which is the whole reason Llama-3-70B (and most modern serving-oriented LLMs) ship with GQA rather than full multi-head attention. **Key takeaway: GQA is not a training-time regularization trick — it is a memory-bandwidth-and-capacity optimization aimed squarely at the decode-side economics this notebook just plotted.**

In [ ]:
# Experiment: quantify the GQA lever and the batching lever the insight above describes.

# (1) GQA's exact effect on KV-cache size: full multi-head attention (K_groups=None -> uses K_heads)
# vs. Llama-3-70B's actual GQA config, at the same 128K context.
full_mha_spec = dict(L_layers=80, K_heads=64, d_head=128, dtype_bytes=2, K_groups=None)
kv_full_mha = kv_cache_bytes(**full_mha_spec, P_tokens=128_000)
kv_gqa = kv_cache_bytes(**kv_spec, P_tokens=128_000)

print("KV-cache at 128K context, per request:")
print(f"  Full multi-head attention (64 KV heads): {kv_full_mha/1e9:6.1f} GB")
print(f"  GQA, 8 KV groups (actual Llama-3-70B):   {kv_gqa/1e9:6.1f} GB")
print(f"  Reduction factor: {kv_full_mha/kv_gqa:.1f}x  (== K_heads / K_groups == {64/8:.1f})\n")

# (2) Batching lever for decode: throughput at batch size B is still capped by memory bandwidth,
# but B requests amortize ONE weight read across B tokens of useful output -- this is why
# continuous batching (not more FLOPs) is the production fix for decode throughput.
for batch in [1, 8, 32, 128]:
    # effective AI improves roughly B-fold since B tokens' worth of output shares one weight read
    effective_ai = decode_ai * batch
    achieved = roofline(effective_ai, PEAK_FLOPS, PEAK_BW)
    frac_of_peak = achieved / PEAK_FLOPS
    print(f"  batch={batch:4d}: effective AI={effective_ai:8.1f} FLOPs/byte, "
          f"{frac_of_peak*100:5.1f}% of peak compute reached")

print("\nBatching pushes decode's operating point rightward along the SAME roofline --")
print("it never needs to leave the memory-bound region to become dramatically more efficient,")
print("which is exactly why continuous batching, not faster tensor cores, is the production lever.")

## Exercise

1. The `speculative_speedup` function implements the Leviathan–Kalai formula for the expected speedup of speculative decoding as a function of the draft model's acceptance rate $\alpha$ and the number of speculated tokens $\gamma$. Sweep $\gamma$ over a range for a few fixed values of $\alpha$ (say 0.5, 0.7, 0.9), plot the resulting speedup curves, and identify — analytically or numerically — the $\gamma$ that maximizes speedup for each $\alpha$. Explain, in terms of the roofline model above, *why* speculative decoding is a memory-bandwidth optimization: which operating point in this notebook's roofline plot does verifying $\gamma$ draft tokens in one pass move decode toward? (See §14.11 for the full derivation of the speedup formula.)

2. Use `quantize_int8_perchannel` and `dequantize` to quantize a random weight matrix shaped like a 70B model's largest linear layer, and recompute `tpot_memory_bound` assuming INT8 weights instead of bf16 (i.e. `dtype_bytes=1` in the `weight_bytes` calculation). Compare the resulting TPOT to the bf16 baseline computed in this notebook, and measure the quantization error (`W - dequantize(quantize_int8_perchannel(W))`) to discuss the accuracy/latency tradeoff this lever represents. (See §14.10 and §15.6 for a fuller treatment of post-training quantization schemes.)